# Chapter 5 — Radar Systems — equations

Standalone, runnable subset of the master `../RF_Equations.ipynb`, scoped to this chapter.
Run top-to-bottom: **Setup**, then this chapter's sections. All functions are verified against the book's worked examples.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

C = 2.99792458e8        # speed of light, m/s
EPS0 = 8.8541878128e-12 # vacuum permittivity, F/m

def wavelength(f_hz):
    return C / f_hz


## 12. Radar Range Equation & RCS — Ch 5 *(future feature)*

Off the indoor one-way path, but useful for a future **radar mode** or a **scattering
reflectivity** model. Key idea: a reflected return is *two-way* Friis → falls as **1/R⁴**
(doubling range = 12 dB, not 6). Reflectivity is **RCS** `σ` (m², or dBsm) — an *electrical*
area independent of physical size.

- `Pr = Pt Gt Gr λ² σ / ((4π)³ R⁴)` (eq 5.8) → `radar_rx_power_dbw()`.
- `Rmax = [Pt G² σ λ² / (Prmin (4π)³)]^(1/4)` (eq 5.9) → `radar_max_range_m()`.
- RCS shapes (Table 5.1): sphere `πr²`, flat plate `4π(lw)²/λ²`, trihedral `4πl⁴/(3λ²)`.
- **Clutter** (§5.4): area backscatter `σ⁰` (RCS/m²), volume `η` (RCS/m³). Clutter area grows
  with R so it falls off slower than a point target — the analog of diffuse wall scatter.


In [ ]:
def radar_rx_power_dbw(pt_dbw, gt_db, gr_db, rcs_m2, wavelength_m, R_m):
    return (pt_dbw + gt_db + gr_db + 10*np.log10(rcs_m2) + 20*np.log10(wavelength_m)
            - 30*np.log10(4*np.pi) - 40*np.log10(R_m))               # eq 5.8

def radar_max_range_m(pt_dbw, g_db, rcs_m2, wavelength_m, pr_min_dbw):
    num = pt_dbw + 2*g_db + 10*np.log10(rcs_m2) + 20*np.log10(wavelength_m)  # eq 5.9
    return 10**((num - (pr_min_dbw + 30*np.log10(4*np.pi)))/40)

def rcs_sphere(r_m):                 return np.pi*r_m**2
def rcs_flat_plate(l_m, w_m, wavelength_m): return 4*np.pi*(l_m*w_m)**2/wavelength_m**2

# Example 5.1: 2 GHz, Pt=0 dBW, G=18 dB, R=2 km, sigma=1 m2, B=50 kHz, F=5 dB
pr = radar_rx_power_dbw(0, 18, 18, 1.0, wavelength(2e9), 2000)
n  = -174 + 10*np.log10(50e3) + 5          # dBm
print(f"Ex 5.1: Pr={pr:.1f} dBW = {pr+30:.1f} dBm, N={n:.0f} dBm, SNR={pr+30-n:.1f} dB (book 6.5)")

# Example 5.2: 10 GHz, Pt=60 dBW, G=28 dB, tau=100us, sigma=1 m2 @ 20 km, Teff=200 K
pr2 = radar_rx_power_dbw(60, 28, 28, 1.0, wavelength(10e9), 20000)
n2  = -204 + 10*np.log10(1/100e-6) + 10*np.log10(1 + 200/290)   # dBW
print(f"Ex 5.2: Pr={pr2:.1f} dBW, N={n2:.1f} dBW, SNR={pr2-n2:.1f} dB (book 42.2)")
